# Ćwiczenie 4.3: Random Forest, czyli wiele drzew, losowość i głosowanie

W poprzednich notebookach budowaliśmy **jedno drzewo decyzyjne**.

Tutaj robimy naturalny kolejny krok: zamiast ufać jednemu drzewu, budujemy wiele drzew i agregujemy ich decyzje.

```text
Decision Tree  = jedno drzewo, jedna struktura reguł
Random Forest  = wiele drzew, losowość, głosowanie/uśrednianie, większa stabilność
```

Pracujemy na problemie churn: **czy klient odejdzie z usługi?**

Plik: **wersja dla studentów**.


## 0. Most z poprzednich notebooków

To, co już umiemy:

```text
Notebook 4.1
cechy binarne -> Gini -> jedno drzewo -> klasa

Notebook 4.2A
cechy liczbowe/punktowe -> próg -> Gini -> jedno drzewo -> klasa

Notebook 4.2B
cechy liczbowe/punktowe -> próg -> MSE/MAE -> jedno drzewo regresyjne -> liczba
```

Teraz dochodzi nowy pomysł:

```text
Notebook 4.3
wiele drzew -> bootstrap + losowanie cech -> głosowanie/uśrednianie -> klasa lub prawdopodobieństwo
```

Ważne: **Random Forest nie jest nowym typem pytania w węźle**.

Pojedyncze drzewa w lesie nadal robią to samo co wcześniej:

```text
cecha binarna = 1?
```

albo:

```text
cecha liczbowa <= próg?
```

Nowe jest to, że:

1. budujemy wiele drzew,
2. każde drzewo dostaje trochę inną próbkę danych,
3. przy splitach drzewo widzi tylko część cech,
4. wynik lasu to agregacja wyników drzew.


### Doprecyzowanie: Bagging a Random Forest

W praktyce warto rozdzielić dwa źródła losowości:

```text
Bagging:
    wiele drzew + bootstrap wierszy

Random Forest:
    wiele drzew + bootstrap wierszy + losowanie podzbioru cech przy splitach
```

Czyli Random Forest można traktować jako mocniejszą wersję baggingu dla drzew. Sam bootstrap sprawia, że drzewa widzą trochę inne obserwacje. Losowanie cech sprawia dodatkowo, że drzewa nie opierają się cały czas na tych samych najsilniejszych zmiennych.

To jest szczególnie ważne, gdy jedna cecha jest bardzo mocna. Bez losowania cech wiele drzew miałoby bardzo podobny korzeń, a las byłby mniej różnorodny.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display
from sklearn.base import clone
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.dummy import DummyClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
)

pd.set_option("display.max_rows", 80)
pd.set_option("display.max_columns", 60)


def make_one_hot_encoder():
    # Działa zarówno w nowszych, jak i starszych wersjach scikit-learn.
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)


def sigmoid(z):
    return 1 / (1 + np.exp(-z))


def is_missing(value):
    # Pomocniczo: sprawdza, czy student zostawił wielokropek.
    return value is Ellipsis


def all_filled(*values):
    return not any(is_missing(value) for value in values)


## 1. Mini-las na małych danych: co jest nowe względem jednego drzewa?

Zaczynamy od małego przykładu, żeby zobaczyć mechanikę Random Forest bez dużej tabeli.

Dane są już uproszczone do cech binarnych:

- `niski_staz`: klient ma krótki staż,
- `duzo_reklamacji`: klient zgłaszał dużo reklamacji,
- `malo_logowan`: klient rzadko się loguje,
- `umowa_miesieczna`: klient ma umowę miesięczną,
- `odejdzie`: etykieta docelowa.

To jest podobne do notebooka 4.1: cechy są już w formie pytań `TAK/NIE`.


In [ ]:
mini_churn = pd.DataFrame({
    "klient_id": [1, 2, 3, 4, 5, 6, 7, 8],
    "niski_staz": [1, 1, 0, 0, 1, 0, 1, 0],
    "duzo_reklamacji": [1, 0, 1, 0, 1, 0, 0, 1],
    "malo_logowan": [1, 1, 0, 0, 0, 0, 1, 1],
    "umowa_miesieczna": [1, 1, 0, 0, 1, 0, 0, 1],
    "odejdzie": ["tak", "tak", "nie", "nie", "tak", "nie", "tak", "nie"],
})

display(mini_churn)


## 2. Nowość 1: bootstrap, czyli losowanie wierszy z powtórzeniami

W pojedynczym drzewie zwykle używaliśmy całego zbioru treningowego.

W Random Forest każde drzewo dostaje własną próbkę danych. Ta próbka jest losowana **z powtórzeniami**.

Przykład dla pierwszego drzewa:

```text
drzewo_1 dostaje klientów: 1, 2, 2, 3, 5, 5, 6, 8
```

Zwróć uwagę:

- klient `2` pojawia się dwa razy,
- klient `5` pojawia się dwa razy,
- niektórzy klienci w ogóle nie trafili do tego drzewa.

Ci klienci, którzy nie trafili do danego drzewa, nazywają się **OOB** — *out-of-bag*.


In [ ]:
bootstrap_drzewo_1 = [1, 2, 2, 3, 5, 5, 6, 8]

# TODO A: uzupełnij po policzeniu ręcznie z listy powyżej.
#
# Pseudokod z kartki:
# 1. Policz, ilu różnych klientów pojawia się w bootstrap_drzewo_1.
# 2. Policz, ile razy występuje klient 2.
# 3. Wypisz klientów z mini_churn, których NIE MA w bootstrap_drzewo_1.
#
# To nie jest jeszcze trenowanie modelu. To tylko sprawdzenie,
# czy rozumiesz losowanie z powtórzeniami.

liczba_unikalnych_klientow = ...
ile_razy_klient_2 = ...
oob_drzewo_1 = ...

print("Liczba unikalnych klientów:", liczba_unikalnych_klientow)
print("Ile razy występuje klient 2:", ile_razy_klient_2)
print("Klienci OOB dla drzewa 1:", oob_drzewo_1)


In [ ]:
# Komórka kontrolna — uruchom po uzupełnieniu TODO A.
# Kod liczy to samo automatycznie, ale celem ćwiczenia jest najpierw policzenie ręczne.

expected_unique = len(set(bootstrap_drzewo_1))
expected_client_2_count = bootstrap_drzewo_1.count(2)
expected_oob = sorted(set(mini_churn["klient_id"]) - set(bootstrap_drzewo_1))

if not all_filled(liczba_unikalnych_klientow, ile_razy_klient_2, oob_drzewo_1):
    print("Najpierw uzupełnij TODO A.")
else:
    print("Oczekiwane wartości:")
    print("- liczba unikalnych klientów:", expected_unique)
    print("- ile razy klient 2:", expected_client_2_count)
    print("- OOB:", expected_oob)
    print()
    print("Czy liczba unikalnych klientów OK?", liczba_unikalnych_klientow == expected_unique)
    print("Czy liczba wystąpień klienta 2 OK?", ile_razy_klient_2 == expected_client_2_count)
    print("Czy lista OOB OK?", sorted(oob_drzewo_1) == expected_oob)


### Trzy drzewa mają trzy różne próbki bootstrapowe

Poniżej widać, że każde drzewo może uczyć się na trochę innym zbiorze.

To właśnie daje różnorodność drzew.


In [ ]:
bootstrap_samples = {
    "drzewo_1": [1, 2, 2, 3, 5, 5, 6, 8],
    "drzewo_2": [1, 1, 4, 4, 5, 6, 7, 8],
    "drzewo_3": [2, 3, 3, 4, 5, 7, 7, 8],
}

bootstrap_summary = []
all_clients = set(mini_churn["klient_id"])

for tree_name, sample_ids in bootstrap_samples.items():
    oob_ids = sorted(all_clients - set(sample_ids))
    bootstrap_summary.append({
        "drzewo": tree_name,
        "próbka_bootstrap": sample_ids,
        "unikalni_klienci_w_próbce": len(set(sample_ids)),
        "klienci_OOB": oob_ids,
    })

display(pd.DataFrame(bootstrap_summary))


## 3. Nowość 2: każde drzewo nie zawsze widzi wszystkie cechy przy splicie

W pojedynczym drzewie przy danym węźle sprawdzaliśmy wszystkie możliwe cechy/progi i wybieraliśmy najlepszy split.

W Random Forest przy każdym splicie losujemy tylko część cech. Przykładowo:

```text
drzewo_1 w korzeniu widzi tylko:
duzo_reklamacji, umowa_miesieczna
```

Jeśli najlepsza globalnie cecha nie została wylosowana, drzewo musi wybrać najlepszą z dostępnych.

To brzmi jak ograniczenie, ale pomaga: drzewa są mniej podobne do siebie.


In [ ]:
cechy_dostepne_w_korzeniu = pd.DataFrame({
    "drzewo": ["drzewo_1", "drzewo_2", "drzewo_3"],
    "cechy_dostępne_w_korzeniu": [
        ["duzo_reklamacji", "umowa_miesieczna"],
        ["niski_staz", "malo_logowan"],
        ["duzo_reklamacji", "umowa_miesieczna"],
    ],
})

display(cechy_dostepne_w_korzeniu)


### Ćwiczenie B: Gini w korzeniu jednego drzewa z lasu

Dla `drzewo_1` używamy próbki bootstrapowej:

```text
[1, 2, 2, 3, 5, 5, 6, 8]
```

W korzeniu `drzewo_1` może porównać tylko dwie cechy:

```text
duzo_reklamacji
umowa_miesieczna
```

Zadanie: policz ważone Gini dla obu splitów i wybierz lepszy split.

To jest dokładnie ten sam mechanizm co wcześniej, tylko liczony na próbce bootstrapowej i tylko dla wylosowanych cech.


In [ ]:
bootstrap_1_df = mini_churn.set_index("klient_id").loc[bootstrap_drzewo_1].reset_index()
display(bootstrap_1_df)


In [ ]:
# TODO B: wpisz wyniki policzone ręcznie lub półręcznie.
#
# Pseudokod dla jednej cechy, np. duzo_reklamacji:
# 1. Podziel bootstrap_1_df na dwie grupy:
#       lewa:  duzo_reklamacji == 0
#       prawa: duzo_reklamacji == 1
# 2. W każdej grupie policz liczbę etykiet "tak" i "nie".
# 3. Dla każdej grupy policz:
#       Gini = 1 - p_tak**2 - p_nie**2
# 4. Policz ważone Gini:
#       (n_lewa / n) * gini_lewa + (n_prawa / n) * gini_prawa
# 5. Powtórz to samo dla umowa_miesieczna.
# 6. Wybierz cechę z mniejszym ważonym Gini.

gini_duzo_reklamacji = ...
gini_umowa_miesieczna = ...
najlepsza_cecha_w_korzeniu = ...

print("Gini dla duzo_reklamacji:", gini_duzo_reklamacji)
print("Gini dla umowa_miesieczna:", gini_umowa_miesieczna)
print("Najlepsza cecha:", najlepsza_cecha_w_korzeniu)


In [ ]:
# Komórka kontrolna — uruchom po uzupełnieniu TODO B.

def gini_from_labels(labels):
    probs = labels.value_counts(normalize=True)
    return 1 - np.sum(probs ** 2)


def weighted_gini_for_binary_feature(df, feature, target_col="odejdzie"):
    rows = []
    total_n = len(df)
    weighted = 0.0

    for value, group in df.groupby(feature):
        group_gini = gini_from_labels(group[target_col])
        weight = len(group) / total_n
        weighted += weight * group_gini
        rows.append({
            "cecha": feature,
            "wartość_cechy": value,
            "n": len(group),
            "tak": int((group[target_col] == "tak").sum()),
            "nie": int((group[target_col] == "nie").sum()),
            "gini_grupy": group_gini,
            "waga": weight,
        })

    return weighted, pd.DataFrame(rows)


score_duzo, details_duzo = weighted_gini_for_binary_feature(bootstrap_1_df, "duzo_reklamacji")
score_umowa, details_umowa = weighted_gini_for_binary_feature(bootstrap_1_df, "umowa_miesieczna")

print("Szczegóły splitu: duzo_reklamacji")
display(details_duzo.round(6))
print("Szczegóły splitu: umowa_miesieczna")
display(details_umowa.round(6))

reference_scores = pd.DataFrame([
    {"cecha": "duzo_reklamacji", "ważone_Gini": score_duzo},
    {"cecha": "umowa_miesieczna", "ważone_Gini": score_umowa},
]).sort_values("ważone_Gini")

print("Podsumowanie:")
display(reference_scores.round(6))

if not all_filled(gini_duzo_reklamacji, gini_umowa_miesieczna, najlepsza_cecha_w_korzeniu):
    print("Najpierw uzupełnij TODO B.")
else:
    print("Czy Gini dla duzo_reklamacji OK?", np.isclose(gini_duzo_reklamacji, score_duzo, atol=1e-3))
    print("Czy Gini dla umowa_miesieczna OK?", np.isclose(gini_umowa_miesieczna, score_umowa, atol=1e-3))
    print("Czy najlepsza cecha OK?", najlepsza_cecha_w_korzeniu == reference_scores.iloc[0]["cecha"])


## 4. Nowość 3: głosowanie wielu drzew

Pojedyncze drzewo zwraca jedną decyzję.

Las zwraca decyzję po agregacji wielu drzew.

Najprostsza intuicja:

```text
drzewo_1 -> tak
drzewo_2 -> nie
drzewo_3 -> tak
drzewo_4 -> tak
drzewo_5 -> nie

większość -> tak
udział głosów tak -> 3/5 = 0.60
```

W praktyce `RandomForestClassifier.predict_proba` w scikit-learn uśrednia prawdopodobieństwa zwracane przez drzewa. Jeżeli liście są czyste, to jest to bardzo bliskie prostemu udziałowi głosów. Jeżeli liście nie są czyste, średnia prawdopodobieństw może się trochę różnić od twardego głosowania `tak/nie`.


In [ ]:
glosy_demo = ["tak", "nie", "tak", "tak", "nie"]

# TODO C: policz wynik lasu dla powyższych pięciu głosów.
#
# Pseudokod:
# 1. Policz, ile razy pojawia się "tak".
# 2. Policz, ile razy pojawia się "nie".
# 3. p_tak = liczba głosów "tak" / liczba wszystkich głosów.
# 4. Jeżeli głosów "tak" jest więcej niż głosów "nie", predykcja = "tak".
#    W przeciwnym razie predykcja = "nie".

liczba_glosow_tak = ...
liczba_glosow_nie = ...
p_tak_demo = ...
predykcja_demo = ...

print("Głosy:", glosy_demo)
print("Liczba głosów tak:", liczba_glosow_tak)
print("Liczba głosów nie:", liczba_glosow_nie)
print("p_tak:", p_tak_demo)
print("Predykcja:", predykcja_demo)


In [ ]:
# Komórka kontrolna — uruchom po uzupełnieniu TODO C.

expected_tak = glosy_demo.count("tak")
expected_nie = glosy_demo.count("nie")
expected_p_tak = expected_tak / len(glosy_demo)
expected_pred = "tak" if expected_tak > expected_nie else "nie"

if not all_filled(liczba_glosow_tak, liczba_glosow_nie, p_tak_demo, predykcja_demo):
    print("Najpierw uzupełnij TODO C.")
else:
    print("Czy liczba głosów tak OK?", liczba_glosow_tak == expected_tak)
    print("Czy liczba głosów nie OK?", liczba_glosow_nie == expected_nie)
    print("Czy p_tak OK?", np.isclose(p_tak_demo, expected_p_tak))
    print("Czy predykcja OK?", predykcja_demo == expected_pred)


## 5. Dane churn

Teraz przechodzimy do większego, syntetycznego zbioru churn.

Dane są syntetyczne, ale mają kontrolowaną logikę:

- krótki staż, dużo reklamacji i umowa miesięczna zwiększają ryzyko odejścia,
- długi staż, większa aktywność, rabat i dłuższa umowa zmniejszają ryzyko odejścia.

W danych są też cechy kategoryczne i braki danych, więc użyjemy `Pipeline` oraz `ColumnTransformer`.


In [ ]:
def make_churn_data(n=320, random_state=42):
    rng = np.random.default_rng(random_state)

    miesiace = rng.integers(1, 60, n)
    reklamacje = rng.poisson(1.1, n)
    logowania = np.round(np.clip(rng.normal(4.8, 1.9, n), 0, None), 1)
    plan = rng.choice(["basic", "standard", "premium"], n, p=[0.48, 0.34, 0.18])
    umowa = rng.choice(["miesieczna", "roczna", "dwuletnia"], n, p=[0.47, 0.35, 0.18])
    rabat = rng.choice(["tak", "nie"], n, p=[0.33, 0.67])

    logit = (
        1.2
        - 0.045 * miesiace
        + 0.55 * reklamacje
        - 0.32 * logowania
        + 0.75 * (umowa == "miesieczna")
        - 0.65 * (umowa == "dwuletnia")
        + 0.35 * (plan == "basic")
        - 0.30 * (plan == "premium")
        - 0.45 * (rabat == "tak")
        + rng.normal(0, 0.55, n)
    )
    p = sigmoid(logit)
    odejdzie = rng.binomial(1, p)

    df = pd.DataFrame({
        "klient_id": range(1, n + 1),
        "miesiace_umowy": miesiace,
        "liczba_reklamacji": reklamacje,
        "srednie_logowania_tyg": logowania,
        "plan": plan,
        "umowa": umowa,
        "rabat": rabat,
        "odejdzie": np.where(odejdzie == 1, "tak", "nie"),
    })

    missing_logins = rng.choice(df.index, size=max(4, n // 25), replace=False)
    missing_plan = rng.choice(df.index, size=max(3, n // 35), replace=False)
    df.loc[missing_logins, "srednie_logowania_tyg"] = np.nan
    df.loc[missing_plan, "plan"] = np.nan

    return df


churn = make_churn_data()

num_cols = ["miesiace_umowy", "liczba_reklamacji", "srednie_logowania_tyg"]
cat_cols = ["plan", "umowa", "rabat"]
target = "odejdzie"
positive_label = "tak"

X = churn.drop(columns=["odejdzie", "klient_id"])
y = churn[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

num_pipe = Pipeline([("imputer", SimpleImputer(strategy="median"))])
cat_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", make_one_hot_encoder()),
])

preprocess = ColumnTransformer([
    ("num", num_pipe, num_cols),
    ("cat", cat_pipe, cat_cols),
])


def evaluate_text_model(name, estimator, X_test, y_test):
    pred = estimator.predict(X_test)

    if hasattr(estimator, "predict_proba"):
        classes = list(estimator.classes_)
        proba = estimator.predict_proba(X_test)[:, classes.index(positive_label)]
        roc_auc = roc_auc_score((y_test == positive_label).astype(int), proba)
    else:
        roc_auc = np.nan

    return {
        "model": name,
        "accuracy": accuracy_score(y_test, pred),
        "precision_tak": precision_score(y_test, pred, pos_label=positive_label, zero_division=0),
        "recall_tak": recall_score(y_test, pred, pos_label=positive_label, zero_division=0),
        "f1_tak": f1_score(y_test, pred, pos_label=positive_label, zero_division=0),
        "roc_auc": roc_auc,
    }


In [ ]:
print("Rozmiar danych:", churn.shape)
print()
print("Pierwsze wiersze:")
display(churn.head())
print("Rozkład klasy docelowej:")
display(churn["odejdzie"].value_counts(normalize=True).rename("proporcja"))
print("Braki danych:")
display(churn.isna().sum())


## 6. Co widzi model po preprocessingu?

Modele drzewiaste w scikit-learn nie przyjmują bezpośrednio tekstów typu `basic`, `roczna`, `tak`.

Dlatego:

- liczby uzupełniamy medianą,
- kategorie uzupełniamy najczęstszą wartością,
- kategorie zamieniamy na kolumny 0/1 przez one-hot encoding.

Po tym etapie drzewo znowu widzi dane w postaci liczbowej: część cech jest ciągła, część to kolumny 0/1.


In [ ]:
prep_preview = clone(preprocess)
prep_preview.fit(X_train)

one_client = X_train.iloc[[0]]
print("Przed preprocessingiem:")
display(one_client)

transformed = prep_preview.transform(one_client)
transformed_df = pd.DataFrame(
    transformed,
    columns=prep_preview.get_feature_names_out(),
    index=one_client.index,
)
print("Po preprocessingiu:")
display(transformed_df)


## 7. Baseline, pojedyncze drzewo i Random Forest

Porównamy trzy modele:

1. **baseline**: zawsze najczęstsza klasa,
2. **pojedyncze drzewo**: jedna struktura reguł,
3. **Random Forest**: wiele drzew z bootstrapem, losowaniem cech i agregacją.

W churn szczególnie patrzymy na `recall_tak`: ilu klientów, którzy faktycznie odejdą, wykrywamy.

Dodajemy też `oob_score` dla Random Forest. To jakość mierzona na obserwacjach, które były OOB dla części drzew.


In [ ]:
baseline = DummyClassifier(strategy="most_frequent")

tree_pipe = Pipeline([
    ("prep", preprocess),
    ("model", DecisionTreeClassifier(
        max_depth=3,
        min_samples_leaf=8,
        random_state=42,
    )),
])

forest_pipe = Pipeline([
    ("prep", preprocess),
    ("model", RandomForestClassifier(
        n_estimators=200,
        max_depth=7,
        min_samples_leaf=4,
        max_features="sqrt",
        bootstrap=True,
        oob_score=True,
        random_state=42,
        n_jobs=1,
    )),
])

baseline.fit(X_train, y_train)
tree_pipe.fit(X_train, y_train)
forest_pipe.fit(X_train, y_train)

comparison = pd.DataFrame([
    evaluate_text_model("Baseline", baseline, X_test, y_test),
    evaluate_text_model("Decision Tree", tree_pipe, X_test, y_test),
    evaluate_text_model("Random Forest", forest_pipe, X_test, y_test),
])

comparison["oob_score_train"] = np.nan
comparison.loc[comparison["model"] == "Random Forest", "oob_score_train"] = forest_pipe.named_steps["model"].oob_score_

comparison.sort_values("f1_tak", ascending=False).round(3)


### Ćwiczenie D

Odpowiedz po obejrzeniu tabeli:

1. Który model ma najwyższe `accuracy`?
2. Który model ma najwyższe `recall_tak`?
3. Czy baseline jest użyteczny w churn, jeśli ma `recall_tak = 0`?
4. Czy Random Forest jest zawsze najlepszy w każdej metryce?
5. Co oznacza `oob_score_train`?


## 8. Głosy drzew dla jednego klienta

Teraz podejrzymy, jak pojedyncze drzewa głosują dla jednego klienta.

Zobaczymy trzy rzeczy:

1. twarde głosy drzew: `tak` / `nie`,
2. udział twardych głosów `tak`,
3. średnie prawdopodobieństwo `tak` zwracane przez drzewa.

W scikit-learn `predict_proba` lasu to średnia prawdopodobieństw z drzew, a nie wyłącznie liczba twardych głosów. Intuicja głosowania nadal jest bardzo pomocna, ale warto znać ten szczegół.


In [ ]:
rf_model = forest_pipe.named_steps["model"]
rf_prep = forest_pipe.named_steps["prep"]

client = X_test.iloc[[0]]
client_t = rf_prep.transform(client)
positive_index = list(rf_model.classes_).index("tak")

raw_votes = [tree.predict(client_t)[0] for tree in rf_model.estimators_]
vote_labels = [rf_model.classes_[int(v)] for v in raw_votes]

tree_proba_tak = np.array([
    tree.predict_proba(client_t)[0, positive_index]
    for tree in rf_model.estimators_
])

print("Klient:")
display(client)
print("Prawdziwa etykieta:", y_test.iloc[0])
print("Predykcja lasu:", forest_pipe.predict(client)[0])
print("Prawdopodobieństwa lasu:")
display(pd.Series(forest_pipe.predict_proba(client)[0], index=forest_pipe.classes_))
print("Twarde głosy drzew:")
display(pd.Series(vote_labels).value_counts())


In [ ]:
# TODO E: policz agregację głosów dla klienta powyżej.
#
# Pseudokod:
# 1. rf_hard_votes_tak = liczba elementów vote_labels równych "tak".
# 2. rf_hard_vote_share_tak = rf_hard_votes_tak / liczba wszystkich drzew.
# 3. rf_avg_tree_proba_tak = średnia z tablicy tree_proba_tak.
#
# Uwaga:
# - rf_hard_vote_share_tak to udział twardych głosów "tak".
# - rf_avg_tree_proba_tak to to, co odpowiada predict_proba w scikit-learn.

rf_hard_votes_tak = ...
rf_hard_vote_share_tak = ...
rf_avg_tree_proba_tak = ...

print("Liczba drzew:", len(vote_labels))
print("Liczba twardych głosów tak:", rf_hard_votes_tak)
print("Udział twardych głosów tak:", rf_hard_vote_share_tak)
print("Średnie prawdopodobieństwo tak z drzew:", rf_avg_tree_proba_tak)
print("predict_proba lasu dla tak:", forest_pipe.predict_proba(client)[0, positive_index])


In [ ]:
# Komórka kontrolna — uruchom po uzupełnieniu TODO E.

expected_hard_votes_tak = int(np.sum(np.array(vote_labels) == "tak"))
expected_hard_vote_share_tak = expected_hard_votes_tak / len(vote_labels)
expected_avg_tree_proba_tak = float(np.mean(tree_proba_tak))
expected_forest_proba_tak = float(forest_pipe.predict_proba(client)[0, positive_index])

if not all_filled(rf_hard_votes_tak, rf_hard_vote_share_tak, rf_avg_tree_proba_tak):
    print("Najpierw uzupełnij TODO E.")
else:
    print("Czy liczba twardych głosów tak OK?", rf_hard_votes_tak == expected_hard_votes_tak)
    print("Czy udział twardych głosów tak OK?", np.isclose(rf_hard_vote_share_tak, expected_hard_vote_share_tak))
    print("Czy średnie prawdopodobieństwo z drzew OK?", np.isclose(rf_avg_tree_proba_tak, expected_avg_tree_proba_tak))
    print("Czy średnia prawdopodobieństw z drzew zgadza się z predict_proba lasu?", np.isclose(expected_avg_tree_proba_tak, expected_forest_proba_tak))


## 9. Ważność cech

Po dopasowaniu `RandomForestClassifier` udostępnia atrybut `feature_importances_`.

W tym modelu jest to **ważność oparta na spadku nieczystości**. Często spotkasz nazwę **MDI** (*mean decrease in impurity*).

Definicja praktyczna:

```text
Dla każdej cechy model sumuje, o ile splity używające tej cechy
zmniejszały nieczystość Gini w drzewach lasu.
```

Dokładniej:

1. każde drzewo ma wiele splitów,
2. split zmniejsza Gini mniej albo bardziej,
3. spadek Gini jest ważony liczbą obserwacji w danym węźle,
4. wkłady z wielu splitów i wielu drzew są sumowane dla każdej cechy,
5. wynik jest normalizowany, więc wartości sumują się do około `1.0`.

Czytamy to tak:

```text
większa wartość = cecha była dla tego modelu bardziej przydatna przy budowaniu podziałów
```

Krótka notatka interpretacyjna: jest to miara predykcyjna i modelowa. Mówi, z czego korzystał wytrenowany las. Nie jest sama w sobie dowodem przyczynowym, bo do przyczynowości potrzebujemy dodatkowego projektu badania albo eksperymentu.

Dwie praktyczne uwagi:

- cechy mocno skorelowane mogą dzielić między siebie ważność,
- ważność MDI bywa korzystniejsza dla cech, które mają więcej możliwych splitów.

Dlatego niżej pokazujemy też `permutation_importance`.


In [ ]:
feature_names = forest_pipe.named_steps["prep"].get_feature_names_out()
importance = pd.DataFrame({
    "cecha_po_transformacji": feature_names,
    "importance": forest_pipe.named_steps["model"].feature_importances_,
}).sort_values("importance", ascending=False)

importance["udział_%"] = 100 * importance["importance"]

print("Suma importance:", round(float(importance["importance"].sum()), 6))
display(importance.head(10).round({"importance": 4, "udział_%": 2}))

top = importance.head(10).iloc[::-1]
plt.figure(figsize=(8, 4))
plt.barh(top["cecha_po_transformacji"], top["importance"])
plt.xlabel("feature_importance / spadek Gini po normalizacji")
plt.title("Random Forest: najważniejsze cechy według spadku Gini")
plt.show()


### Dodatkowa kontrola: permutation importance

`feature_importances_` patrzy do środka modelu i sprawdza, które cechy dawały dobre splity.

`permutation_importance` zadaje inne pytanie:

```text
Co stanie się z jakością modelu, jeśli jedną cechę losowo przemieszamy?
```

Jeżeli po przemieszaniu cechy `f1_tak` mocno spada, to znaczy, że model korzystał z informacji zawartej w tej cesze przy predykcji na zbiorze testowym.

Różnica:

```text
feature_importances_      -> miara wewnętrzna, oparta na splitach w drzewach
permutation_importance    -> miara zewnętrzna, oparta na spadku jakości predykcji
```


In [ ]:
from sklearn.inspection import permutation_importance
from sklearn.metrics import make_scorer

f1_tak_scorer = make_scorer(f1_score, pos_label="tak", zero_division=0)

perm = permutation_importance(
    forest_pipe,
    X_test,
    y_test,
    scoring=f1_tak_scorer,
    n_repeats=10,
    random_state=42,
    n_jobs=1,
)

permutation_table = pd.DataFrame({
    "cecha_oryginalna": X_test.columns,
    "spadek_f1_tak_po_permutacji": perm.importances_mean,
    "odchylenie": perm.importances_std,
}).sort_values("spadek_f1_tak_po_permutacji", ascending=False)

display(permutation_table.round(4))


## 10. Próg decyzyjny w Random Forest

`predict_proba` zwraca prawdopodobieństwo klasy `tak`, czyli ocenę ryzyka churnu.

Sama klasa `tak/nie` powstaje dopiero po ustawieniu progu:

```text
p_odejdzie_tak >= próg  -> przewidujemy "tak"
p_odejdzie_tak <  próg  -> przewidujemy "nie"
```

Domyślny próg `0.50` jest tylko punktem startowym. W churn często sprawdzamy także niższe progi, bo firma może chcieć wykryć więcej klientów zagrożonych odejściem.

W tabeli poniżej pokazujemy nie tylko `precision_tak`, `recall_tak` i `f1_tak`, ale też konkretne liczby z macierzy pomyłek: `TP`, `FP`, `FN`, `TN`. Pełne przypomnienie metryk jest w appendixie po ćwiczeniu F.


In [ ]:
proba_tak = forest_pipe.predict_proba(X_test)[:, list(forest_pipe.classes_).index("tak")]


def threshold_metrics_table_rf(y_true, proba_positive, thresholds):
    """Tabela metryk dla różnych progów klasy 'tak'."""
    rows = []
    for threshold in thresholds:
        pred_thr = np.where(proba_positive >= threshold, "tak", "nie")

        # labels=["tak", "nie"] daje układ:
        # [[TP, FN],
        #  [FP, TN]]
        cm = confusion_matrix(y_true, pred_thr, labels=["tak", "nie"])
        tp = int(cm[0, 0])
        fn = int(cm[0, 1])
        fp = int(cm[1, 0])
        tn = int(cm[1, 1])

        rows.append({
            "threshold": float(threshold),
            "pred_tak": int(tp + fp),
            "TP_zlapany_churn": tp,
            "FP_falszywy_alarm": fp,
            "FN_przeoczony_churn": fn,
            "TN_poprawne_nie": tn,
            "accuracy": accuracy_score(y_true, pred_thr),
            "precision_tak": precision_score(y_true, pred_thr, pos_label="tak", zero_division=0),
            "recall_tak": recall_score(y_true, pred_thr, pos_label="tak", zero_division=0),
            "f1_tak": f1_score(y_true, pred_thr, pos_label="tak", zero_division=0),
        })
    return pd.DataFrame(rows)


threshold_table = threshold_metrics_table_rf(
    y_test,
    proba_tak,
    thresholds=[0.25, 0.35, 0.50, 0.65],
)

display(threshold_table.round(3))

plt.figure(figsize=(8, 4))
plt.plot(threshold_table["threshold"], threshold_table["precision_tak"], marker="o", label="precision_tak")
plt.plot(threshold_table["threshold"], threshold_table["recall_tak"], marker="o", label="recall_tak")
plt.plot(threshold_table["threshold"], threshold_table["f1_tak"], marker="o", label="f1_tak")
plt.xlabel("próg dla klasy tak")
plt.ylabel("wartość metryki")
plt.title("Random Forest: wpływ progu decyzyjnego")
plt.legend()
plt.grid(True)
plt.show()


### Jak wybrać próg w ćwiczeniu?

W praktyce próg ustala się przez koszt decyzji. W tym ćwiczeniu przyjmujemy konkretną regułę:

```text
Chcemy wysoki recall_tak, ale precision_tak nie powinno spaść poniżej 0.40.
```

Czyli:

1. odrzucamy progi, przy których za dużo alarmów jest nietrafionych,
2. spośród pozostałych wybieramy ten, który łapie najwięcej prawdziwych churnów,
3. przy remisie patrzymy na `f1_tak`.


In [ ]:
def suggest_threshold_for_churn(table, min_precision=0.40):
    """Wybiera próg według reguły: precision >= min_precision, potem najwyższy recall."""
    candidates = table[table["precision_tak"] >= min_precision].copy()

    if candidates.empty:
        candidates = table.copy()
        sort_cols = ["f1_tak", "recall_tak", "precision_tak"]
    else:
        sort_cols = ["recall_tak", "f1_tak", "precision_tak"]

    return candidates.sort_values(sort_cols, ascending=False).iloc[0]


min_precision_w_cwiczeniu = 0.40
kandydaci_progowe = threshold_table[threshold_table["precision_tak"] >= min_precision_w_cwiczeniu]

print(f"Kandydaci z precision_tak >= {min_precision_w_cwiczeniu}:")
display(kandydaci_progowe.sort_values(["recall_tak", "f1_tak"], ascending=False).round(3))

sugerowany_wiersz = suggest_threshold_for_churn(threshold_table, min_precision=min_precision_w_cwiczeniu)
print("Próg sugerowany przez przyjętą regułę:")
display(sugerowany_wiersz.to_frame().T.round(3))


### Ćwiczenie F: wybór progu decyzyjnego

Uzupełnij komórkę poniżej.

Cel biznesowy:

```text
Chcemy złapać możliwie dużo klientów, którzy naprawdę odejdą,
ale nie chcemy zejść poniżej precision_tak = 0.40.
```

W praktyce:

```text
1. Popatrz na tabelę kandydatów powyżej.
2. Wybierz próg z wysokim recall_tak.
3. Sprawdź, ilu klientów trafia do kontaktu: pred_tak = TP + FP.
4. Sprawdź, ile churnów nadal przeoczysz: FN_przeoczony_churn.
5. Napisz krótkie uzasadnienie decyzji.
```

Jeśli potrzebujesz przypomnienia metryk, zajrzyj do appendixu pod ćwiczeniem.


In [ ]:
# TODO F: wybierz próg decyzyjny na podstawie tabeli i reguły biznesowej.
#
# Wskazówka:
# - threshold_table zawiera metryki dla progów 0.25, 0.35, 0.50, 0.65.
# - sugerowany_wiersz pokazuje próg wybrany przez prostą regułę z komórki wyżej.
# - możesz użyć tej sugestii albo świadomie wybrać inny próg i uzasadnić dlaczego.

wybrany_prog = ...
uzasadnienie = "..."

if not all_filled(wybrany_prog) or uzasadnienie.strip() in {"", "..."}:
    print("Uzupełnij wybrany_prog oraz krótkie uzasadnienie.")
else:
    wybrany_wiersz = threshold_table[np.isclose(threshold_table["threshold"], wybrany_prog)]
    if wybrany_wiersz.empty:
        print("Ten próg nie występuje w tabeli. Wybierz jeden z progów pokazanych w threshold_table.")
    else:
        print("Wybrany próg:", wybrany_prog)
        print("Uzasadnienie:", uzasadnienie)
        display(wybrany_wiersz.round(3))


### Notatka po Ćwiczeniu F

Próg decyzyjny nie zmienia samego modelu ani jego prawdopodobieństw. Zmienia tylko decyzję biznesową:

```text
kogo oznaczamy jako klienta ryzykownego?
```

W churn wybór progu często opisuje się przez koszt dwóch błędów:

```text
FP = kontaktujemy klienta niepotrzebnie
FN = nie zauważamy klienta, który naprawdę odchodzi
```


## Appendix do Ćwiczenia F: przypomnienie metryk klasyfikacji

Ten appendix jest przypomnieniem starszej wiedzy potrzebnej do wyboru progu. Nie jest nowym elementem Random Forest, tylko narzędziem do interpretacji decyzji.

Dla klasy `tak`, czyli „klient odejdzie”, używamy czterech liczności:

| Symbol | Znaczenie w churn |
|---|---|
| `TP` | klient naprawdę odszedł i model przewidział `tak` |
| `FP` | klient nie odszedł, ale model przewidział `tak` |
| `FN` | klient odszedł, ale model przewidział `nie` |
| `TN` | klient nie odszedł i model przewidział `nie` |

Najważniejsze wzory:

```text
precision_tak = TP / (TP + FP)
recall_tak    = TP / (TP + FN)
f1_tak        = 2 * precision_tak * recall_tak / (precision_tak + recall_tak)
accuracy      = (TP + TN) / (TP + FP + FN + TN)
```

Interpretacja:

```text
precision_tak
    Spośród klientów oznaczonych jako „odejdzie”, ilu naprawdę odchodzi?

recall_tak
    Spośród klientów, którzy naprawdę odchodzą, ilu wykryliśmy?

f1_tak
    Jeden wynik łączący precision i recall. Przydatny, gdy nie chcemy patrzeć tylko na jedną stronę kompromisu.
```

Przy obniżaniu progu dla klasy `tak` zwykle dzieje się to:

```text
próg niższy -> więcej predykcji „tak” -> wyższy recall_tak, ale więcej FP
próg wyższy -> mniej predykcji „tak” -> wyższa precision_tak, ale więcej FN
```

Dlatego w Ćwiczeniu F nie pytamy „który próg jest absolutnie najlepszy?”, tylko:

```text
który próg jest najlepszy dla przyjętej reguły biznesowej?
```


## 11. Mini-eksperyment z parametrami lasu

Sprawdzamy, jak `n_estimators`, `max_depth` i `max_features` wpływają na wynik.

Intuicyjnie:

- `n_estimators`: liczba drzew; więcej drzew zwykle stabilizuje wynik, ale zwiększa koszt obliczeń,
- `max_depth`: maksymalna głębokość drzewa; większa głębokość może dać bardziej złożone reguły,
- `max_features`: ile cech drzewo może rozważać przy splicie; mniejsza wartość zwiększa losowość i różnorodność drzew.


W ostatnim wariancie tabeli `max_features=None` oznacza, że drzewo może przy splicie rozważać wszystkie cechy. To jest zachowanie bardziej podobne do zwykłego baggingu drzew. Warianty z `max_features="sqrt"` są bardziej typowe dla Random Forest, bo zwiększają różnorodność drzew.


In [ ]:
settings = [
    {"n_estimators": 20, "max_depth": 3, "max_features": "sqrt"},
    {"n_estimators": 80, "max_depth": 3, "max_features": "sqrt"},
    {"n_estimators": 80, "max_depth": 7, "max_features": "sqrt"},
    {"n_estimators": 200, "max_depth": 7, "max_features": "sqrt"},
    {"n_estimators": 200, "max_depth": None, "max_features": "sqrt"},
    {"n_estimators": 200, "max_depth": 7, "max_features": None},
]

rows = []
for params in settings:
    pipe = Pipeline([
        ("prep", preprocess),
        ("model", RandomForestClassifier(
            **params,
            min_samples_leaf=4,
            bootstrap=True,
            oob_score=True,
            random_state=42,
            n_jobs=1,
        )),
    ])
    pipe.fit(X_train, y_train)
    row = evaluate_text_model(
        f"RF n={params['n_estimators']}, depth={params['max_depth']}, max_features={params['max_features']}",
        pipe,
        X_test,
        y_test,
    )
    row["oob_score_train"] = pipe.named_steps["model"].oob_score_
    rows.append({**params, **row})

pd.DataFrame(rows).drop(columns=["model"]).round(3)


## 12. Jednozdaniowy schemat algorytmu

Random Forest dla klasyfikacji działa tak:

```text
Dla każdego drzewa:
    1. wylosuj próbkę bootstrapową z danych treningowych,
    2. buduj drzewo jak wcześniej,
    3. przy każdym splicie rozważ tylko losowy podzbiór cech,
    4. wybierz najlepszy split według Gini/entropii.

Dla nowego klienta:
    1. przepuść klienta przez każde drzewo,
    2. zbierz głosy lub prawdopodobieństwa,
    3. zagreguj wyniki,
    4. zwróć klasę i/lub prawdopodobieństwo.
```


## Pytania końcowe

1. Co dokładnie zmienia Random Forest względem pojedynczego drzewa?
2. Co oznacza bootstrap i dlaczego w próbce mogą powtarzać się ci sami klienci?
3. Co oznacza OOB i dlaczego może być wewnętrznym sprawdzeniem jakości modelu?
4. Po co losujemy tylko część cech przy każdym splicie?
5. Dlaczego `predict_proba` w lesie można rozumieć jako agregację opinii wielu drzew?
6. Dlaczego `accuracy` nie wystarcza przy churn?
7. Co zmienia obniżenie progu dla klasy `tak`?
8. Jak jest liczona `feature_importances_` w Random Forest i dlaczego nie jest to automatycznie wniosek przyczynowy?


## Łącznik do Notebooka 4.4: dlaczego boosting to nie „kolejny las”

Po Random Forest łatwo pomyśleć, że każdy kolejny mocny model drzewiasty to po prostu „więcej drzew”. To nie jest najlepsza intuicja.

```text
Random Forest:
    drzewa są budowane niezależnie
    każde drzewo ma własną próbkę bootstrapową
    wynik = głosowanie / średnia

Gradient Boosting:
    drzewa są budowane sekwencyjnie
    nowe drzewo patrzy na błędy aktualnego modelu
    wynik = suma kolejnych poprawek
```

Dlatego `n_estimators` znaczy trochę co innego w obu modelach:

| Parametr | Random Forest | Gradient Boosting |
|---|---|---|
| `n_estimators` | liczba niezależnych drzew głosujących razem | liczba kolejnych kroków/poprawek |
| Za dużo drzew | zwykle stabilniej, ale wolniej | może pomóc, ale przy złym `learning_rate` może przeuczać |
| Główna idea | redukcja wariancji przez uśrednianie | redukcja błędów przez kolejne poprawki |

To jest główny most do Gradient Boostingu i XGBoost.